In [2]:
import sys
from pathlib import Path

# Path setup
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Seed 
from src.utils.seed import set_seed
set_seed(42)

# Imports
import torch
from src.data_operations.dataset import build_dataloaders
from src.data_operations.transforms import get_train_transforms, get_val_transforms

Seed set to 42


In [3]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA GeForce RTX 4070 Ti SUPER


In [4]:
from src.models.transformer.vit import build_vit

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vit_model = build_vit(num_classes=6)
vit_model = vit_model.to(device)

vit_model.print_trainable_layers()

params = vit_model.count_parameters()
print(f"Trainable: {params['trainable']:,}")
print(f"Total:     {params['total']:,}")


Layer                Status
--------------------------------
conv_proj            frozen
encoder              frozen
heads                trainable
Trainable: 4,614
Total:     85,803,270


In [5]:
from src.data_operations.dataset import build_dataloaders
from src.data_operations.transforms import get_train_transforms, get_val_transforms

splits_root = project_root / "data" / "splits"

train_loader, val_loader, test_loader = build_dataloaders(
    root            = splits_root,
    train_transform = get_train_transforms(),
    val_transform   = get_val_transforms(),
    batch_size      = 32,
    num_workers     = 4,
    pin_memory      = True,
)
print(train_loader)

In [ ]:
num_epochs = 10
from src.training.train import train, evaluate

train(
    model = vit_model, 
    train_loader = train_loader, 
    val_loader = val_loader, 
    num_epochs = num_epochs)

Epoch 1/20
  Train — loss: 0.6570  acc: 0.8732
  Val   — loss: 0.2810  acc: 0.9815
Epoch 2/20
  Train — loss: 0.1487  acc: 0.9856
  Val   — loss: 0.1555  acc: 0.9852
Epoch 3/20
  Train — loss: 0.0893  acc: 0.9904
  Val   — loss: 0.1065  acc: 0.9963
Epoch 4/20
  Train — loss: 0.0627  acc: 0.9968
  Val   — loss: 0.0835  acc: 0.9963


In [ ]:
test_loss, test_acc = evaluate(vit_model, test_loader)

loss: 0.0133  acc: 1.0000


In [ ]:
from src.interpretability.attention_viz import AttentionRollout
import random

idx    = random.randint(0, len(test_loader.dataset) - 1)

rollout = AttentionRollout(vit_model)

sample  = test_loader.dataset[idx]
boxes   = test_loader.dataset.get_boxes(sample["image_path"])
classes = test_loader.dataset.classes

rollout.visualize(sample, classes, boxes=boxes)

ModuleNotFoundError: No module named 'src'